In [1]:
# from paddleocr import PPStructureV3

In [2]:
# pipeline = PPStructureV3(
#     # engine="transformers", #intelegent model is used 
#     use_formula_recognition=False, #it will not detect foumual 
#     wired_table_structure_recognition_model_name="RT-DETR-L_wired_table_cell_det", # this model is used to detect the content without table lines 
#     lang="en",
#     use_doc_orientation_classify=False,
#     use_doc_unwarping=False,
#     use_textline_orientation=False,
#     device=paddle.device.get_device(),
#     cpu_threads=8 
# )


In [3]:
# output = pipeline.predict(img)

In [4]:
# output

In [5]:
# !nvcc --version

In [6]:
# !nvidia-smi

In [7]:
# print("CUDA built:", paddle.device.is_compiled_with_cuda())
# print("GPU count:", paddle.device.cuda.device_count())
# print("Device:", paddle.device.get_device())

# display version of python used 

In [8]:
!py --version

Python 3.14.2


# import the libary

In [9]:
from paddleocr import PaddleOCR 
import paddle
import cv2
import numpy as np
import os
from llama_cpp import Llama
import json

D:\Anuj_S_Jain\Data\Code\Project\Society_MCE_QT\Society_MCE\Prototype\.venv\lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.3.0)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(
D:\Anuj_S_Jain\Data\Code\Project\Society_MCE_QT\Society_MCE\Prototype\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Checking connectivity to the model hosters, this may take a while. To bypass this check, set `DISABLE_MODEL_SOURCE_CHECK` to `True`.
D:\Anuj_S_Jain\Data\Code\Project\Society_MCE_QT\Society_MCE\Prototype\.venv\lib\site-packages\paddle\utils\cpp_extension\extension_utils.py:718: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccac

# device used

In [10]:
paddle.device.get_device()

'cpu'

In [11]:
paddle.device.cuda.device_count()

0

# loading image path

In [12]:
i="4"
image_path = f"D:/Anuj_S_Jain/Data/Code/Project/Society_MCE_QT/Society_MCE/Prototype/training data/diff_img/{i}.jpg"
image_path

'D:/Anuj_S_Jain/Data/Code/Project/Society_MCE_QT/Society_MCE/Prototype/training data/diff_img/4.jpg'

# read img

In [13]:
print(os.path.exists(image_path))

True


In [14]:
img = cv2.imread(image_path)

# run paddleocr on img 

In [15]:
if (img is None) :
    print(f"{image_path} this is incorrect no img found")
else:
    # setting tread 
    # os.environ["OMP_NUM_THREADS"] = "8"
    # os.environ["MKL_NUM_THREADS"] = "8"
    # applying ocr on the image
    ocr = PaddleOCR(use_doc_orientation_classify=False, 
        use_doc_unwarping=False, 
        use_textline_orientation=False,
        lang='en',
        device=paddle.device.get_device(),
        cpu_threads=8 
       )
    result = ocr.predict(img)

Creating model: ('PP-OCRv5_server_det', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\Anuj S Jain\.paddlex\official_models\PP-OCRv5_server_det`.
Creating model: ('en_PP-OCRv5_mobile_rec', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\Anuj S Jain\.paddlex\official_models\en_PP-OCRv5_mobile_rec`.


In [16]:
result

[{'input_path': None,
  'page_index': None,
  'doc_preprocessor_res': {'output_img': array([[[255, ..., 255],
           ...,
           [255, ..., 255]],
   
          ...,
   
          [[255, ..., 255],
           ...,
           [255, ..., 255]]], shape=(1580, 1998, 3), dtype=uint8)},
  'dt_polys': [array([[58,  4],
          ...,
          [56, 44]], shape=(4, 2), dtype=int16),
   array([[552,  19],
          ...,
          [551,  54]], shape=(4, 2), dtype=int16),
   array([[55, 41],
          ...,
          [55, 81]], shape=(4, 2), dtype=int16),
   array([[1304,   34],
          ...,
          [1303,   74]], shape=(4, 2), dtype=int16),
   array([[1497,   40],
          ...,
          [1496,   76]], shape=(4, 2), dtype=int16),
   array([[1628,   46],
          ...,
          [1628,   82]], shape=(4, 2), dtype=int16),
   array([[1754,   45],
          ...,
          [1753,   77]], shape=(4, 2), dtype=int16),
   array([[ 58,  99],
          ...,
          [ 58, 135]], shape=(4, 2), 

In [17]:
points=result[0]["dt_polys"][0].tolist()

In [18]:
points[0]

[58, 4]

In [25]:
points=contours[3].tolist()

In [26]:
# point coordinates
point_xy=6
x = points[point_xy][0][0]
y = points[point_xy][0][1]

# draw point
cv2.circle(
    img,
    (x, y),      # center point
    10,           # radius
    (0, 255, 255), # color (B,G,R)
    -1           # filled circle
)

cv2.imwrite(f'./output/opencv/points{i}.jpeg', img)

True

# extracting the table from the img 

In [27]:
if (img is None) :
    print(f"{image_path} this is incorrect no img found")
else:
   
    # converting img to gray
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    # cv2.imwrite(f'./output/opencv/gray{i}.jpeg', gray)

    # splits the image and clahe 
    # C – Contrast
    # L – Limited
    # A – Adaptive
    # H – Histogram
    # E – Equalization
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    gray_clahe = clahe.apply(gray)    
    # cv2.imwrite(f'./output/opencv/gray_clahe{i}.jpeg', gray_clahe)
    
    # Better threshold (adaptive handles uneven lighting)
    thresh = cv2.adaptiveThreshold(
        gray_clahe, 255,
        cv2.ADAPTIVE_THRESH_MEAN_C,
        cv2.THRESH_BINARY_INV,
        15, 10
    )
     
    
    # --- Horizontal lines ---
    horizontal_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (60,1))
    horizontal = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, horizontal_kernel)
    # cv2.imwrite(f'./output/opencv/horizontal{i}.jpeg', horizontal)
    
    # connect broken horizontal lines
    horizontal = cv2.dilate(horizontal, np.ones((2,2),np.uint8), iterations=2)
    # cv2.imwrite(f'./output/opencv/horizontal_dilated{i}.jpeg', horizontal)
    
    # --- Vertical lines ---
    vertical_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (1,60))
    vertical = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, vertical_kernel)
    # cv2.imwrite(f'./output/opencv/vertical{i}.jpeg', vertical)
     
    # connect broken vertical lines
    vertical = cv2.dilate(vertical, np.ones((2,2),np.uint8), iterations=2)
    # cv2.imwrite(f'./output/opencv/vertical_dilate{i}.jpeg', vertical)
    
    
    # Combine
    boxes = cv2.add(horizontal, vertical)
    # cv2.imwrite(f'./output/opencv/boxes_combine{i}.jpeg', boxes)
    
    # Final closing to join gaps
    kernel = np.ones((10,10), np.uint8)
    boxes = cv2.morphologyEx(boxes, cv2.MORPH_CLOSE, kernel, iterations=2)
    cv2.imwrite(f'./output/opencv/boxes_fill_gap{i}.jpeg', boxes)  
    # cv2.imwrite(f'./output/opencv/new/boxes_fill_gap{i}.jpeg', boxes)  

    # kernel = np.ones((2,2), np.uint8)
    # erode = cv2.erode(boxes, kernel, iterations=2)
    # cv2.imwrite(f'./output/opencv/erode{i}.jpeg', erode)
    
    # adding border
    h, w = img.shape[:2]
    border=cv2.rectangle(boxes, (0,0), (w,h), (255,255,255), 3)
    cv2.imwrite(f'./output/opencv/bordered{i}.jpeg', border)
    # cv2.imwrite(f'./output/opencv/new/bordered{i}.jpeg', border)
    
    # Find contours
    contours, _ = cv2.findContours(border, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
    output = img.copy()
    cv2.imwrite(f'./output/opencv/output{i}.jpeg', output)
    # cv2.imwrite(f'./output/opencv/new/output{i}.jpeg', output)

# find and draw the counter

In [28]:
# output = img.copy()

# # for index,cnt in enumerate(contours):
# #     x,y,w,h = cv2.boundingRect(cnt)
# #     # filter noise
# #     if w > 100 and h > 100:
# cv2.drawContours(output, contours, -1, (0,255,0), 1)
# cv2.imwrite(f'./output/opencv/output{i}.jpeg', output)

In [29]:
# contours

# Downscaling the paddle ocr border to small

#### ref code 

In [30]:
# ref code 
# new_poly = []
# for poly in result[0]["rec_polys"]:
#     # convert to float for calculation
#     poly = np.array(poly, dtype=np.float32)
#     cv2.polylines(output, [np.array(poly, dtype=np.int32)],isClosed=True, color=(255, 0, 0),thickness=2) 

#     x_min, y_min = np.min(poly, axis=0)
#     x_max, y_max = np.max(poly, axis=0)
#     w = x_max - x_min
#     h = y_max - y_min

#     # adaptive shrink
#     shrink_factor_h = max(0.6, min(0.9, 1 - 20/w))
#     shrink_factor_v = max(0.6, min(0.9, 1 - 20/h))
#     # print(poly)
    
#     # compute center
#     center_x = np.mean(poly[:, 0])
#     center_y = np.mean(poly[:, 1])

#     poly_box=[]
#     for (x,y) in poly:
#         x_new=int(center_x+(x-center_x)*shrink_factor_h)
#         y_new=int(center_y+(y-center_y)*shrink_factor_v)
#         poly_box.append([x_new,y_new])
#     new_poly.append(np.array(poly_box, dtype=np.int16))
#     cv2.polylines(output, [np.array(poly_box, dtype=np.int32)],isClosed=True, color=(0, 255, 255),thickness=2) 
# cv2.imwrite(f'./output/opencv/scaling{i}.jpeg', output)

In [31]:
def downscaling(poly_list):
    poly_downscaling=[]
    for poly in poly_list:
        # convert to float for calculation
        poly = np.array(poly, dtype=np.float32)
        cv2.polylines(output, [np.array(poly, dtype=np.int32)],isClosed=True, color=(255, 0, 0),thickness=2) 
    
        x_min, y_min = np.min(poly, axis=0)
        x_max, y_max = np.max(poly, axis=0)
        w = x_max - x_min
        h = y_max - y_min
    
        # adaptive shrink
        shrink_factor_h = max(0.6, min(0.9, 1 - 20/w))
        shrink_factor_v = max(0.6, min(0.9, 1 - 20/h))
        # print(poly)
        
        # compute center
        center_x = np.mean(poly[:, 0])
        center_y = np.mean(poly[:, 1])
    
        poly_box=[]
        for (x,y) in poly:
            x_new=int(center_x+(x-center_x)*shrink_factor_h)
            y_new=int(center_y+(y-center_y)*shrink_factor_v)
            poly_box.append([x_new,y_new])
        # cv2.polylines(output, [np.array(poly_box, dtype=np.int32)],isClosed=True, color=(0, 255, 255),thickness=2) 
        # cv2.imwrite(f'./output/opencv/scaling{i}.jpeg', output)
        poly_downscaling.append(np.array(poly_box, dtype=np.int16))
    return poly_downscaling


# make an array by combining the ploy points , text , and confident scores

In [32]:
result[0]['rec_texts']

['SI',
 'Description of Goods',
 'No.',
 'Quantity',
 'Rate',
 'per',
 'Amount',
 '1',
 'Books',
 '5,000 Nos',
 'Record Book',
 '38.00',
 'Nos',
 '1,90,000.00',
 'Size-1/4th Pages-4+96',
 'Cover-3o0gsm Duplex Board',
 'With Multicolor Printing',
 '& Matt Lamination',
 'Inner 60gsm Maplitho',
 'Single Color Printing',
 'Finishing-Perfect Binding',
 '2',
 'Books',
 '2,500 Nos',
 'Record Book',
 '52.00',
 'Nos',
 '1,30,000.00',
 'Size-1/4th Pages-4+144',
 'Cover 300gsm Dup',
 'Board',
 'With Multicolor Printing',
 '& Matt Lamination',
 'Inner-60gsm Maplitho Single',
 'Color Printing',
 'Finishing-Perfect Binding',
 '3,20,000.00',
 'CGST (Output) 9%',
 '9',
 '%',
 '28,800.00',
 'Cow Bu',
 'SGST (Output) 9%',
 '9',
 '%',
 '28,800.00',
 '415342',
 'ou10226',
 'Total',
 '7,500 Nos',
 '3,77,600.00',
 'Amount Chargeable (in words)',
 'E. &O.E',
 'INR Three Lakh Seventy Seven Thousand Six Hundred Only',
 'HSN/SAC',
 'Taxable',
 'Central Tax',
 'State Tax',
 'Total']

In [33]:
downscaling_poly=downscaling(result[0]['dt_polys'])
ocr_result = [(poly , text , scores)
    for poly , text , scores  in 
        zip(downscaling_poly,
            result[0]['rec_texts'],
            result[0]['rec_scores'])]

In [34]:
# downscaling_poly[0].tolist()

In [35]:
ocr_result

[(array([[66, 12],
         ...,
         [65, 36]], shape=(4, 2), dtype=int16),
  'SI',
  0.9975608587265015),
 (array([[565,  27],
         ...,
         [564,  48]], shape=(4, 2), dtype=int16),
  'Description of Goods',
  0.9885762929916382),
 (array([[64, 49],
         ...,
         [64, 73]], shape=(4, 2), dtype=int16),
  'No.',
  0.99658203125),
 (array([[1313,   43],
         ...,
         [1312,   67]], shape=(4, 2), dtype=int16),
  'Quantity',
  0.9999382495880127),
 (array([[1506,   47],
         ...,
         [1506,   69]], shape=(4, 2), dtype=int16),
  'Rate',
  0.9998124241828918),
 (array([[1638,   53],
         ...,
         [1638,   74]], shape=(4, 2), dtype=int16),
  'per',
  0.999910295009613),
 (array([[1763,   52],
         ...,
         [1763,   71]], shape=(4, 2), dtype=int16),
  'Amount',
  0.9999079704284668),
 (array([[ 64, 106],
         ...,
         [ 64, 127]], shape=(4, 2), dtype=int16),
  '1',
  0.9998181462287903),
 (array([[117, 107],
         ...,
    

# to get the overlapping percentage 

In [ ]:
# import numpy as np

# def rectangle_overlap_percentage(rectA, rectB ):
#     # """
#     # rectA and rectB: list of 4 points [[x1,y1],[x2,y2],[x3,y3],[x4,y4]]
#     # Can also be numpy arrays of shape (4,2)
#     # Returns overlap % relative to rectA
#     # """
#     # Convert to numpy arrays
#     rectA = np.array(rectA)
#     rectB = np.array(rectB)
    
#     # Get bounding box
#     x_min_A, x_max_A = rectA[:,0].min(), rectA[:,0].max()
#     y_min_A, y_max_A = rectA[:,1].min(), rectA[:,1].max()
    
#     x_min_B, x_max_B = rectB[:,0].min(), rectB[:,0].max()
#     y_min_B, y_max_B = rectB[:,1].min(), rectB[:,1].max()
    
#     # Overlap width & height
#     overlap_width = max(0, min(x_max_A, x_max_B) - max(x_min_A, x_min_B))
#     overlap_height = max(0, min(y_max_A, y_max_B) - max(y_min_A, y_min_B))
    
#     # Intersection area
#     intersection_area = overlap_width * overlap_height
    
#     # Area of rectangle A
#     area_A = (x_max_A - x_min_A) * (y_max_A - y_min_A)
    
#     # Overlap %
#     overlap_percentage = (intersection_area / area_A) * 100 if area_A > 0 else 0

#     return overlap_percentage

In [ ]:
# contours[1]

In [ ]:
# rectA = np.array((contours[1].reshape(-1,2)).tolist())
# rectA

In [ ]:
# result[0]['rec_polys'][8].tolist()

In [ ]:
# for ocr in ocr_result:
#     ocr[0]
# np.array(ocr[2])

In [ ]:
def rectangle_overlap_percentage(paddleocr_box, counters):
    #rectA = paddleocr text box
    #rectB = counter of the table
    # rectA and rectB: list of 4 points [[x1,y1],[x2,y2],[x3,y3],[x4,y4]]
    #how much of a is in b
    # Convert to numpy arrays
    rectA_rectB_merge=[]
    counter=[]
    box=[]
    text=[]
    # for counter_i in range(len(counters)):
    for counter_poly in counters:
        ocr_box=[]
        ocr_text=[]
        # for paddleocr_i in range(len(paddleocr_box)):
        for paddleocr_poly in paddleocr_box:
            # rectA_org = paddleocr_box[paddleocr_i][0].tolist()
            # rectB_org = counters[counter_i][:, 0, :].tolist()
            
            # rectA_org = paddleocr_poly.tolist()
            # rectB_org = counter_poly.tolist()
                    
            # rectA = np.array(rectA_org)
            # rectB = np.array(rectB_org)
            
            rectA = np.array(paddleocr_poly[0].tolist())
            rectB = np.array((counter_poly.reshape(-1,2)).tolist())
            
            # Get bounding box
            x_min_A, x_max_A = rectA[:,0].min(), rectA[:,0].max()
            y_min_A, y_max_A = rectA[:,1].min(), rectA[:,1].max()
            
            x_min_B, x_max_B = rectB[:,0].min(), rectB[:,0].max()
            y_min_B, y_max_B = rectB[:,1].min(), rectB[:,1].max()
            
            # Overlap width & height
            overlap_width = max(0, min(x_max_A, x_max_B) - max(x_min_A, x_min_B))
            overlap_height = max(0, min(y_max_A, y_max_B) - max(y_min_A, y_min_B))
            
            # Intersection area
            intersection_area = overlap_width * overlap_height
            
            # Area of rectangle A
            area_A = (x_max_A - x_min_A) * (y_max_A - y_min_A)
            
            # Overlap %
            overlap_percentage = (intersection_area / area_A) * 100 if area_A > 0 else 0

            #append if overlapping is more 
            if(overlap_percentage > 90 and paddleocr_poly[2] > 0): #removed confident score to 0
                ocr_box.append(paddleocr_poly[0])
                ocr_text.append(paddleocr_poly[1])
                
                
            # else:
            #     continue
        if (ocr_box):
            # rectA_rectB_merge.append((counter_poly,ocr_box,ocr_text)) # adding [] to use it as counter point as an indiviudal points
            counter.append(counter_poly)
            box.append(ocr_box)
            text.append(ocr_text)
        # rectA_rectB_merge.append(ocr_box)
    return counter,box,text


In [ ]:
counter_only,box_only,text_only=rectangle_overlap_percentage(ocr_result,contours)

In [ ]:
len(counter_only[0])

In [ ]:
len(box_only[0])

In [ ]:
len(text_only[0])

In [ ]:
text_only

In [ ]:
# merged_counter_ocr[7][0] # gives counter

In [ ]:
# merged_counter_ocr[1][1] # gives poly of text

In [ ]:
# merged_counter_ocr[1][2] # gives text

In [ ]:
# len(contours)

In [ ]:
# len(grouped_counter_and_text)

In [ ]:
# ans[2][1]

In [ ]:
# np.array(grouped_counter_and_text[3][1]).tolist()

# display the counter and text poly box on the image [for testingg]

In [ ]:
img_test = cv2.imread(f'./output/opencv/output{i}.jpeg')
# for cnt_no in range(len(box_only)):
#     # cv2.drawContours(img_test, ans[cnt_no][1], -1, (255,0,0), 3)
#     for poly_ocr_box in box_only[cnt_no]:
#         cv2.polylines(img_test, [np.array(poly_ocr_box, dtype=np.int32)],isClosed=True, color=(255,255,0 ),thickness=3)
#     cv2.imwrite(f'./output/opencv/test{i}.jpeg', img_test)



cnt_no=1
cv2.drawContours(img_test, counter_only[cnt_no], -1, (255,0,255), 3)
for poly_ocr_box in box_only[cnt_no]:
    cv2.polylines(img_test, [np.array(poly_ocr_box, dtype=np.int32)],isClosed=True, color=(255,0,0 ),thickness=3)
cv2.imwrite(f'./output/opencv/test{i}.jpeg', img_test)


In [ ]:
# for i in range(len(contours)):
#     for j in range(len(result1[0]['rec_polys'])):
#         overlap = rectangle_overlap_percentage(result1[0]['rec_polys'][i].tolist(),contours[j][:, 0, :].tolist())
#         if(overlap > 50):
#             print(f"Overlap % of Rectangle A: {overlap:.2f}%")
#             cv2.drawContours(output, contours, j, (255,255,0), 2)
#             cv2.polylines(output, [np.array(result1[0]['rec_polys'][i], dtype=np.int32)],isClosed=True, color=(0, 0, 255),thickness=2)
# cv2.imwrite('./output/opencv/imagetry.jpeg', output)

# arranging the contours horizontally and vertically 

#### find the center of x and y

In [ ]:
def get_center_from_rect(rect):
    x, y, w, h = rect
    cx = x + w / 2
    cy = y + h / 2
    return cx, cy

#### grouping counter vertically 

In [ ]:
def group_contours_into_lines_h(contours, x_threshold=20):
    # Step 1: convert contours to bounding boxes 
    rects = [(cv2.boundingRect(cnt),cnt) for cnt in contours]
    #[x1,y1,w1,h1]

    # Step 2: compute centers
    rects_with_centers = [(rect, get_center_from_rect(rect),cnt) for rect,cnt in rects]
    #[[x1,y1,w1,h1],[cx,yc],[org cnt]]

    # Step 3: sort top → bottom
    rects_with_centers.sort(key=lambda r: r[1][0])  # sort by cx

    lines = []
    current_line = []

    for rect, (cx, cy),cnt in rects_with_centers:
        #if the current_line list is emptry append the first element 
        if not current_line:
            current_line.append((rect, cx, cy,cnt))
            continue
            
        #get the last cx
        _, prev_cx,_, _ = current_line[-1]

        # this is the buffer area where all belong to honrisontal 
        if abs(cx - prev_cx) < x_threshold:
            current_line.append((rect, cx, cy,cnt))
        #start as a new line 
        else:
            lines.append(current_line)
            current_line = [(rect, cx, cy,cnt)]

    if current_line:
        lines.append(current_line)

    # Step 4: sort each line left → right
    result = []
    result_org = []
    for line in lines:
        line.sort(key=lambda r: r[2])  # sort by cy
        result.append([rect for rect, _, _,_ in line])
        result_org.append([cnt for _, _, _,cnt in line])

    return result,result_org
    # return lines

#### grouping counter horizontal

In [ ]:
def group_contours_into_lines_v(contours, y_threshold=20):
    # Step 1: convert contours to bounding boxes 
    rects = [(cv2.boundingRect(cnt),cnt) for cnt in contours]
    #[x1,y1,w1,h1]

    # Step 2: compute centers
    rects_with_centers = [(rect, get_center_from_rect(rect),cnt) for rect,cnt in rects]
    #[[x1,y1,w1,h1],[cx,yc],[org cnt]]

    # Step 3: sort top → bottom
    rects_with_centers.sort(key=lambda r: r[1][1])  # sort by cy

    lines = []
    current_line = []

    for rect, (cx, cy),cnt in rects_with_centers:
        #if the current_line list is emptry append the first element 
        if not current_line:
            current_line.append((rect, cx, cy,cnt))
            continue
            
        #get the last cy
        _, _, prev_cy,_ = current_line[-1]

        # this is the buffer area where all belong to honrisontal 
        if abs(cy - prev_cy) < y_threshold:
            current_line.append((rect, cx, cy,cnt))
        #start as a new line 
        else:
            lines.append(current_line)
            current_line = [(rect, cx, cy,cnt)]

    if current_line:
        lines.append(current_line)

    # Step 4: sort each line left → right
    result = []
    result_org = []
    for line in lines:
        line.sort(key=lambda r: r[1])  # sort by cy
        result.append([rect for rect, _, _,_ in line])
        result_org.append([cnt for _, _, _,cnt in line])

    return result,result_org
    # return lines

In [ ]:
ans_h,ans_org_h = group_contours_into_lines_h(counter_only)
ans_v,ans_org_v = group_contours_into_lines_v(counter_only)

In [ ]:
for i_cnt in range(len(ans_org_h)):
    # output = cv2.imread(f'./output/opencv/output{i}.jpeg')
    # cv2.drawContours(output, ans_org_h[i_cnt], -1, (255,255,0), 3)
    # cv2.imwrite(f'./output/opencv/output{i_cnt}.jpeg', output)
    print(len(ans_org_h[i_cnt]))

In [ ]:
for i_cnt in range(len(ans_org_v)):
    # output = cv2.imread(f'./output/opencv/output{i}.jpeg')
    # cv2.drawContours(output, ans_org_v[i_cnt], -1, (255,255,0), 3)
    # cv2.imwrite(f'./output/opencv/output{i_cnt}.jpeg', output)
    print(len(ans_org_v[i_cnt]))

# getting the header of the table 

In [1]:
def get_heading_col(number_col,ans_org_v):
    for i_cnt in range(len(ans_org_v)):
        if(len(ans_org_v[i_cnt]) == number_col):
            return ans_org_v[i_cnt]

In [2]:
table_header=get_heading_col(6,ans_org_v)

NameError: name 'ans_org_v' is not defined

In [ ]:
table_header

# match the table header with text and text box 

In [ ]:
def counter_matching (matching_counters,counter_only,box_only,text_only):
    box=[]
    text=[]
    for matching_counter in matching_counters:
        for index,counter in enumerate(counter_only):
            if (matching_counter.shape == counter.shape) and np.all(matching_counter == counter):
                box.append(box_only[index])
                text.append(text_only[index])
    return box,text
            
        

In [ ]:
matched_cnt=counter_matching(table_header,counter_only,box_only,text_only)

In [ ]:
matched_cnt[1]

In [ ]:
ans_org_v[0][1]

# header info extraction from vertical align 

In [ ]:
ans_org_v[0]

In [ ]:
def counter_matching_header(header_indexs,table_header,ans_org_h,counter_only,box_only,text_only):
    matched_cnt=[]
    for header_index in header_indexs:
        for index,counter_group in enumerate(ans_org_h):
            for counter in counter_group:
                if (table_header[header_index].shape == counter.shape) and np.all(table_header[header_index] == counter):
                    
                    matched_cnt.append(ans_org_h[index])

            #     box.append(box_only[index])
            #     text.append(text_only[index])
    return matched_cnt

In [ ]:
header_info=counter_matching_header([1,2,5],table_header,ans_org_h,counter_only,box_only,text_only)

In [ ]:
ref_info=counter_matching(header_info[0],counter_only,box_only,text_only)

In [ ]:
ref_info[1]

# arranging the text box of paddleocr horizontally

In [20]:
def get_center(box):
    #[[x1,y1],[x2,y2],[x3,y3],[x4,y4]]
    # print(box[0],box[1],box[2],box[3])
    x_coords = [p[0] for p in box]
    print(sum(x_coords)/4)
    y_coords = [p[1] for p in box]
    return sum(x_coords) / 4, sum(y_coords) / 4


In [39]:
def group_polys_by_horizontal_position(
    ocr_result,
    buffer_percent=0.05,
    min_score=0.90
):

    processed = []

    # Step 1: Extract data from PaddleOCR result
    for poly, text, score in ocr_result:

        if score < min_score:
            continue

        poly = np.array(poly)

        # points:
        # [top-left, top-right, bottom-right, bottom-left]

        x1 = poly[0][0]
        x2 = poly[1][0]

        # mean x
        mean_x = (x1 + x2) / 2

        # approximate width
        width = abs(x2 - x1)

        # center y for top-bottom sorting
        mean_y = np.mean(poly[:, 1])

        processed.append({
            "poly": poly.tolist(),
            "text": text,
            "score": score,
            "mean_x": mean_x,
            "mean_y": mean_y,
            "width": width
        })

    # Step 2: Sort left-to-right
    processed.sort(key=lambda x: x["mean_x"])

    grouped = []
    current_group = []

    for item in processed:

        if not current_group:
            current_group.append(item)
            continue

        prev = current_group[-1]

        avg_width = (item["width"] + prev["width"]) / 2

        # dynamic threshold
        dynamic_buffer = avg_width * buffer_percent

        threshold = avg_width + dynamic_buffer

        # same column check
        if abs(item["mean_x"] - prev["mean_x"]) <= threshold:

            current_group.append(item)

        else:

            grouped.append(current_group)
            current_group = [item]

    if current_group:
        grouped.append(current_group)

    # Step 3: Sort each group top-to-bottom
    final_polys = []
    final_texts = []

    for group in grouped:

        group.sort(key=lambda x: x["mean_y"])

        final_polys.append([g["poly"] for g in group])
        final_texts.append([g["text"] for g in group])

    return final_polys, final_texts

In [40]:
poly,text =group_polys_by_vertical_position(ocr_result)

In [41]:
text

[['No.', 'SI', 'Description of Goods', 'Quantity', 'Rate', 'per', 'Amount'],
 ['1', 'Books'],
 ['Record Book', '5,000 Nos', '38.00', 'Nos', '1,90,000.00'],
 ['Size-1/4th Pages-4+96'],
 ['Cover-3o0gsm Duplex Board'],
 ['With Multicolor Printing'],
 ['& Matt Lamination'],
 ['Inner 60gsm Maplitho'],
 ['Single Color Printing'],
 ['Finishing-Perfect Binding'],
 ['2', 'Books'],
 ['Record Book', '2,500 Nos', '52.00', 'Nos', '1,30,000.00'],
 ['Size-1/4th Pages-4+144'],
 ['Cover 300gsm Dup', 'Board'],
 ['With Multicolor Printing'],
 ['& Matt Lamination'],
 ['Inner-60gsm Maplitho Single'],
 ['Color Printing'],
 ['Finishing-Perfect Binding'],
 ['3,20,000.00'],
 ['415342',
  'CGST (Output) 9%',
  'SGST (Output) 9%',
  '9',
  '9',
  '%',
  '28,800.00',
  '28,800.00'],
 ['Amount Chargeable (in words)',
  'INR Three Lakh Seventy Seven Thousand Six Hundred Only',
  'Total',
  '7,500 Nos',
  '3,77,600.00',
  'E. &O.E'],
 ['HSN/SAC', 'Taxable', 'Central Tax', 'State Tax', 'Total']]

In [ ]:
poly,text = group_boxes_into_lines(ocr_result)

In [ ]:
# poly

In [ ]:
text

In [ ]:
# ocr_result

# arranging the text box of paddleocr vertical

In [67]:
def get_center(box):
    #[[x1,y1],[x2,y2],[x3,y3],[x4,y4]]
    # print(box[0],box[1],box[2],box[3])
    x_coords = [p[0] for p in box]
    print(sum(x_coords)/4)
    y_coords = [p[1] for p in box]
    return sum(x_coords) / 4, sum(y_coords) / 4
# def get_mean_y(box):

#     y1 = box[0][1]
#     y3 = box[2][1]

#     mean_y = (y1 + y3) / 2

#     return mean_y

In [70]:

def group_boxes_into_lines_v(boxes, x_threshold=5):
    # Step 1: compute centers
    box_centers = [(box, get_center(box),text,score) for box,text,score in boxes]

    # Step 2: sort by Y (top to bottom)
    box_centers.sort(key=lambda b: b[1][1])
    print("=============================================================")
    for box in box_centers:
        print(box[2])
    print("=============================================================")


    lines = []
    current_line = []

    for box, (cx, cy) ,text ,score in box_centers:
        if score >= 0.9: #90% above confident score is only used for text 
            if not current_line:
                current_line.append((box, cx, cy, text, score))
                continue
    
            _, _, prev_cy, _, _ = current_line[-1]
    
            # Step 3: check if same line
            y_width_avg=((box[2][1]-box[0][1])+(box[3][1]-box[1][1]))/2
            if abs(cy - prev_cy) < (y_width_avg+x_threshold):
                current_line.append((box, cx, cy, text, score))
            else:
                lines.append(current_line)
                current_line = [(box, cx, cy, text, score)]
        else:
            continue

    if current_line:
        lines.append(current_line)

    # Step 4: sort each line by X (left to right)
    sorted_lines = []
    sorted_lines_text = []
    for line in lines:
        line.sort(key=lambda b: b[2])  # sort by center y
        sorted_lines.append([b[0] for b in line])
        sorted_lines_text.append([b[3] for b in line])

    return sorted_lines,sorted_lines_text

In [71]:
poly,text = group_boxes_into_lines_v(ocr_result)
text

79.5
684.5
77.5
1373.0
1538.75
1661.0
1817.5
73.0
169.75
1373.0
250.5
1568.75
1657.75
1825.25
324.0
342.5
326.75
285.75
314.75
306.5
340.5
69.5
166.5
1365.75
246.0
1562.5
1647.75
1817.25
329.0
274.0
484.25
323.0
282.0
338.25
250.5
340.5
1809.25
1101.5
1587.0
1633.0
1825.25
229.25
1102.0
1587.5
1631.0
1824.75
371.0
432.0
1220.25
1350.25
1802.25
231.25
1836.75
590.25
482.5
1044.75
1283.75
1578.25
1818.25
SI
Description of Goods
Quantity
Rate
No.
Amount
per
1
Books
5,000 Nos
38.00
Nos
Record Book
1,90,000.00
Size-1/4th Pages-4+96
Cover-3o0gsm Duplex Board
With Multicolor Printing
& Matt Lamination
Inner 60gsm Maplitho
Single Color Printing
Finishing-Perfect Binding
2
Books
2,500 Nos
52.00
Nos
1,30,000.00
Record Book
Size-1/4th Pages-4+144
Cover 300gsm Dup
Board
With Multicolor Printing
& Matt Lamination
Inner-60gsm Maplitho Single
Color Printing
Finishing-Perfect Binding
3,20,000.00
Cow Bu
CGST (Output) 9%
9
28,800.00
%
415342
SGST (Output) 9%
%
9
28,800.00
ou10226
Total
7,500 Nos
3,77,60

[['SI', 'Description of Goods', 'Quantity', 'Rate', 'No.', 'Amount', 'per'],
 ['1', 'Books'],
 ['5,000 Nos', '38.00', 'Nos', 'Record Book', '1,90,000.00'],
 ['Size-1/4th Pages-4+96'],
 ['Cover-3o0gsm Duplex Board'],
 ['With Multicolor Printing'],
 ['& Matt Lamination'],
 ['Inner 60gsm Maplitho'],
 ['Single Color Printing'],
 ['Finishing-Perfect Binding'],
 ['2', 'Books'],
 ['2,500 Nos', '52.00', 'Nos', '1,30,000.00', 'Record Book'],
 ['Size-1/4th Pages-4+144'],
 ['Cover 300gsm Dup', 'Board'],
 ['With Multicolor Printing'],
 ['& Matt Lamination'],
 ['Inner-60gsm Maplitho Single'],
 ['Color Printing'],
 ['Finishing-Perfect Binding'],
 ['3,20,000.00'],
 ['CGST (Output) 9%',
  '9',
  '28,800.00',
  '%',
  '415342',
  'SGST (Output) 9%',
  '9',
  '28,800.00'],
 ['Total',
  '7,500 Nos',
  '3,77,600.00',
  'Amount Chargeable (in words)',
  'E. &O.E',
  'INR Three Lakh Seventy Seven Thousand Six Hundred Only'],
 ['HSN/SAC', 'Taxable', 'Central Tax', 'State Tax', 'Total']]

# llm integration

In [72]:
 result[0]['rec_texts']

['SI',
 'Description of Goods',
 'No.',
 'Quantity',
 'Rate',
 'per',
 'Amount',
 '1',
 'Books',
 '5,000 Nos',
 'Record Book',
 '38.00',
 'Nos',
 '1,90,000.00',
 'Size-1/4th Pages-4+96',
 'Cover-3o0gsm Duplex Board',
 'With Multicolor Printing',
 '& Matt Lamination',
 'Inner 60gsm Maplitho',
 'Single Color Printing',
 'Finishing-Perfect Binding',
 '2',
 'Books',
 '2,500 Nos',
 'Record Book',
 '52.00',
 'Nos',
 '1,30,000.00',
 'Size-1/4th Pages-4+144',
 'Cover 300gsm Dup',
 'Board',
 'With Multicolor Printing',
 '& Matt Lamination',
 'Inner-60gsm Maplitho Single',
 'Color Printing',
 'Finishing-Perfect Binding',
 '3,20,000.00',
 'CGST (Output) 9%',
 '9',
 '%',
 '28,800.00',
 'Cow Bu',
 'SGST (Output) 9%',
 '9',
 '%',
 '28,800.00',
 '415342',
 'ou10226',
 'Total',
 '7,500 Nos',
 '3,77,600.00',
 'Amount Chargeable (in words)',
 'E. &O.E',
 'INR Three Lakh Seventy Seven Thousand Six Hundred Only',
 'HSN/SAC',
 'Taxable',
 'Central Tax',
 'State Tax',
 'Total']

In [73]:
chr(10).join(text[0])

'SI\nDescription of Goods\nQuantity\nRate\nNo.\nAmount\nper'

In [74]:
table_text = ""
# table_text=[]
for row in text:
    table_text += " | ".join(row) + "\n"
    # print(table_text)
print(table_text)

SI | Description of Goods | Quantity | Rate | No. | Amount | per
1 | Books
5,000 Nos | 38.00 | Nos | Record Book | 1,90,000.00
Size-1/4th Pages-4+96
Cover-3o0gsm Duplex Board
With Multicolor Printing
& Matt Lamination
Inner 60gsm Maplitho
Single Color Printing
Finishing-Perfect Binding
2 | Books
2,500 Nos | 52.00 | Nos | 1,30,000.00 | Record Book
Size-1/4th Pages-4+144
Cover 300gsm Dup | Board
With Multicolor Printing
& Matt Lamination
Inner-60gsm Maplitho Single
Color Printing
Finishing-Perfect Binding
3,20,000.00
CGST (Output) 9% | 9 | 28,800.00 | % | 415342 | SGST (Output) 9% | 9 | 28,800.00
Total | 7,500 Nos | 3,77,600.00 | Amount Chargeable (in words) | E. &O.E | INR Three Lakh Seventy Seven Thousand Six Hundred Only
HSN/SAC | Taxable | Central Tax | State Tax | Total



In [75]:
# from llama_cpp import Llama

In [76]:
llm = Llama(
    model_path="llama3-Q5_K_M.gguf",
    n_ctx=2048,
    n_threads=8,
    verbose=False  
)

llama_context: n_ctx_seq (2048) < n_ctx_train (131072) -- the full capacity of the model will not be utilized


In [77]:
# prompt = f"""
# You are an expert receipt parser.

# Your task:
# Extract all products from OCR text.

# Rules:
# - A product may span multiple lines
# - Merge lines belonging to same item
# - Extract:
#   - product name
#   - quantity
#   - price
# - Ignore noise text
# - Extract total separately

# Return ONLY valid JSON like:

# {{
#   "items": [
#     {{"product": "", "quantity": "", "price": 0}}
#   ],
#   "total": 0
# }}

# OCR TEXT:
# {chr(10).join( result[0]['rec_texts'])}
# """

# best prompt till now

In [78]:

prompt = f"""
You are a receipt extraction API.

Extract receipt items from OCR text.

Return ONLY valid JSON.
No markdown.
No explanation.
No extra text.
No notes.
No python code

JSON schema:
{{
  "items": [
    {{
      "product": "string",
      "quantity": "string",
      "price": 0
    }}
  ],
  "total_quantity": 0,
  "total_price": 0
}}

Rules:
- Extract ALL products
- Merge multiline products
- Ignore noise
- Extract quantity separately
- Extract total separately
- Price must be numeric

OCR:
{table_text}
"""

In [79]:
prompt = f"""
You are a JSON extraction API.

Extract receipt items from OCR text.

Return EXACTLY ONE valid JSON object.

DO NOT:
- explain
- summarize
- shorten product names
- return multiple JSONs
- add markdown
- add notes

IMPORTANT:
- Preserve FULL product descriptions exactly
- NEVER replace product names with generic names like "Books"
- Merge multiline product descriptions into one line
- Keep all specifications in product text
- Ignore OCR noise

SCHEMA:
{{
  "items": [
    {{
      "product": "string",
      "quantity": "string",
      "price": 0
    }}
  ],
  "total_quantity": "string",
  "total_price": 0
}}

OCR TEXT:
{table_text}

OUTPUT JSON ONLY:
"""

In [80]:
# prompt = f"""
# You are a JSON API.

# Extract receipt items from OCR text.

# STRICT RULES:
# - Return ONLY ONE valid JSON object
# - Do NOT repeat output
# - Do NOT add markdown
# - Do NOT add explanations
# - Do NOT add text before or after JSON
# - Do NOT invent products
# - Ignore subtotal/total rows as products
# - Output must start with {{
# - Output must end with }}

# Schema:
# {{
#   "items": [
#     {{
#       "product": "string",
#       "quantity": "string",
#       "price": 0
#     }}
#   ],
#   "total_quantity": 0,
#   "total_price": 0
# }}

# OCR:
# {chr(10).join(result[0]['rec_texts'])}
# """

In [81]:
# import json
# prompt = f"""
# You are a JSON API.

# Extract invoice items.

# Rules:
# - Ignore tax rows
# - Ignore totals
# - Merge multiline descriptions
# - Return valid JSON only

# Input:
# {json.dumps(rows, indent=2)}

# Return:
# {{
#   "items": [
#     {{
#       "product": "string",
#       "quantity": "string",
#       "price": 0
#     }}
#   ],
#   "total_quantity": 0,
#   "total_price": 0
# }}
# """

In [82]:
output = llm(prompt, max_tokens=300, stop=["\n\n{", "\nJSON", "```"])

In [83]:
print(output["choices"][0]["text"])

[
  {
    "product": "Books, Size-1/4th Pages-4+96 Cover-3o0gsm Duplex Board With Multicolor Printing & Matt Lamination Inner 60gsm Maplitho Single Color Printing Finishing-Perfect Binding",
    "quantity": "5,000 Nos",
    "price": 38.00
  },
  {
    "product": "Books, Size-1/4th Pages-4+144 Cover 300gsm Dup Board With Multicolor Printing & Matt Lamination Inner-60gsm Maplitho Single Color Printing Finishing-Perfect Binding",
    "quantity": "2,500 Nos",
    "price": 52.00
  }
]




In [84]:
output

{'id': 'cmpl-6032eb7e-c76c-4b05-94c0-f83d07688809',
 'object': 'text_completion',
 'created': 1779636160,
 'model': 'llama3-Q5_K_M.gguf',
 'choices': [{'text': '[\n  {\n    "product": "Books, Size-1/4th Pages-4+96 Cover-3o0gsm Duplex Board With Multicolor Printing & Matt Lamination Inner 60gsm Maplitho Single Color Printing Finishing-Perfect Binding",\n    "quantity": "5,000 Nos",\n    "price": 38.00\n  },\n  {\n    "product": "Books, Size-1/4th Pages-4+144 Cover 300gsm Dup Board With Multicolor Printing & Matt Lamination Inner-60gsm Maplitho Single Color Printing Finishing-Perfect Binding",\n    "quantity": "2,500 Nos",\n    "price": 52.00\n  }\n]\n\n',
   'index': 0,
   'logprobs': None,
   'finish_reason': 'stop'}],
 'usage': {'prompt_tokens': 445,
  'completion_tokens': 154,
  'total_tokens': 599}}